# NB04 — Blood–Myocardium Contrast Metrics

**Physics rationale:** Good bSSFP requires high blood–myocardium contrast. Blood appears bright,
myocardium darker but uniform. Off-resonance banding degrades this contrast.

**Metrics:** `cnr`, `contrast_ratio`, `michelson` contrast, `cnr_normalised`

**Selector:** `argmax(cnr)` vs GT

**Runs on:** Local CPU

## 0. Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

Dependencies OK


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

DATA_ROOT  = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH    = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
CACHE_PATH = 'outputs/seg_masks_cache.pkl'
OUT_DIR    = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE    = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]
print('Config OK')

Config OK


In [3]:
from utils.dicom_loader import load_all_patients
from utils.gt_loader import load_gt
from utils.seg_wrapper import load_seg_cache

all_data  = load_all_patients(DATA_ROOT, verbose=False)
gt_dict   = load_gt(GT_PATH)
seg_cache = load_seg_cache(CACHE_PATH)
print(f'Loaded {len(all_data)} patients')

Loaded 26 patients


## 1. Compute Contrast Metrics

In [4]:
def align_mask(mask, target_shape):
    th, tw = target_shape[:2]
    mh, mw = mask.shape[:2]
    if (mh, mw) == (th, tw): return mask.astype(bool)
    if (mh, mw) == (tw, th): return mask.T.astype(bool)
    from PIL import Image as PILImage
    pil = PILImage.fromarray(mask.astype(np.uint8) * 255)
    pil = pil.resize((tw, th), PILImage.NEAREST)
    return (np.asarray(pil) > 127)

def background_pixels(gray):
    H, W = gray.shape
    c = max(10, H // 8)
    return np.concatenate([
        gray[:c, :c].ravel(), gray[:c, -c:].ravel(),
        gray[-c:, :c].ravel(), gray[-c:, -c:].ravel()
    ])

records = []
for pid in sorted(all_data):
    if pid in EXCLUDE: continue
    data = all_data[pid]
    seg  = seg_cache.get(pid)
    if seg is None: continue

    for i, (img, hz) in enumerate(zip(data['images'], data['offsets_hz'])):
        gray = img.astype(np.float32)
        if gray.max() > 1.0: gray /= gray.max()
        myo_mask  = align_mask(seg.myocardium_mask, gray.shape)
        pool_mask = align_mask(seg.lv_pool_mask,    gray.shape)
        myo_px    = gray[myo_mask]
        pool_px   = gray[pool_mask]
        bg_px     = background_pixels(gray)

        mu_myo   = float(myo_px.mean())  if myo_px.size  > 0 else 0.0
        mu_blood = float(pool_px.mean()) if pool_px.size > 0 else 0.0
        sigma_bg = max(float(bg_px.std()), 1e-6)

        cnr            = (mu_blood - mu_myo) / sigma_bg
        contrast_ratio = mu_blood / max(mu_myo, 1e-6)
        michelson      = (mu_blood - mu_myo) / max(mu_blood + mu_myo, 1e-6)

        records.append({
            'patient_id': pid, 'image_no': i+1, 'offset_hz': hz,
            'mu_myo': round(mu_myo, 6), 'mu_blood': round(mu_blood, 6),
            'cnr': round(cnr, 4), 'contrast_ratio': round(contrast_ratio, 4),
            'michelson': round(michelson, 4),
        })

df = pd.DataFrame(records)

def minmax_norm(x):
    r = x.max() - x.min()
    return (x - x.min()) / r if r > 1e-9 else x * 0

df['cnr_norm'] = df.groupby('patient_id')['cnr'].transform(minmax_norm)
df.to_csv(f'{OUT_DIR}/contrast_metrics.csv', index=False)
print(f'Computed {len(df)} rows. Saved contrast_metrics.csv')
df.head(13)

Computed 324 rows. Saved contrast_metrics.csv


,patient_id,image_no,offset_hz,mu_myo,mu_blood,cnr,contrast_ratio,michelson,cnr_norm
0,Patient-BEV,1,-150.0,0.095261,0.195271,1.7351,2.0499,0.3442,0.496344
1,Patient-BEV,2,-125.0,0.068615,0.210382,2.2659,3.0661,0.5081,0.836818
2,Patient-BEV,3,-100.0,0.064495,0.167638,1.7740,2.5992,0.4443,0.521296
3,Patient-BEV,4,-75.0,0.062884,0.142771,1.3636,2.2704,0.3885,0.258050
4,Patient-BEV,5,-50.0,0.060800,0.127933,1.1133,2.1041,0.3557,0.097498
5,Patient-BEV,6,-25.0,0.061270,0.121102,0.9752,1.9765,0.3281,0.008916
6,Patient-BEV,7,0.0,0.062780,0.123643,0.9770,1.9695,0.3265,0.010071
7,Patient-BEV,8,25.0,0.067708,0.142211,1.1786,2.1004,0.3549,0.139384
8,Patient-BEV,9,50.0,0.070436,0.163180,1.4882,2.3167,0.3970,0.337973
9,Patient-BEV,10,75.0,0.057915,0.178449,2.0107,3.0812,0.5100,0.673124


## 2. CNR Curves per Patient

In [5]:
from utils.vis_utils import plot_metric_curves
import matplotlib.pyplot as plt

for metric_col, title in [
    ('cnr',            'CNR (Blood–Myo Contrast-to-Noise Ratio)'),
    ('contrast_ratio', 'Contrast Ratio (mu_blood / mu_myo)'),
    ('michelson',      'Michelson Contrast'),
]:
    metric_dict = {
        pid: df[df['patient_id']==pid].sort_values('image_no')[metric_col].tolist()
        for pid in sorted(df['patient_id'].unique())
    }
    fig = plot_metric_curves(
        metric_dict=metric_dict, offsets_hz=OFFSETS_13, gt_dict=gt_dict, title=title,
        save_path=f'{OUT_DIR}/contrast_{metric_col}_curves.png',
    )
    plt.show()

C:\Users\Hp\AppData\Local\Temp\ipykernel_26188\2293862518.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_26188\2293862518.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_26188\2293862518.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. CNR Heatmap & Blood vs Myo Scatter

In [6]:
from utils.vis_utils import plot_heatmap
import matplotlib.pyplot as plt

patient_ids_sorted = sorted(df['patient_id'].unique())
n = len(patient_ids_sorted)
matrix = np.zeros((n, 13))
for row_i, pid in enumerate(patient_ids_sorted):
    sub = df[df['patient_id']==pid].sort_values('image_no')
    vals = sub['cnr'].values
    matrix[row_i, :len(vals)] = vals

fig = plot_heatmap(
    data_matrix=matrix, patient_ids=patient_ids_sorted, offsets_hz=OFFSETS_13,
    gt_dict=gt_dict, title='CNR Heatmap — GT box = green',
    cbar_label='CNR', cmap='coolwarm',
    save_path=f'{OUT_DIR}/contrast_cnr_heatmap.png',
)
plt.show()

# Blood vs myo scatter at GT image
fig, ax = plt.subplots(figsize=(7, 5))
for pid in df['patient_id'].unique():
    sub = df[df['patient_id']==pid]
    gt_no = gt_dict[pid]['image_no']
    gt_row = sub[sub['image_no']==gt_no]
    if not gt_row.empty:
        ax.scatter(gt_row['mu_myo'].values[0], gt_row['mu_blood'].values[0],
                   color='#69f0ae', s=50, alpha=0.85)
vals = np.linspace(0, 1, 100)
ax.plot(vals, vals, 'r--', alpha=0.4, label='no contrast line')
ax.set_xlabel('mu_myo'); ax.set_ylabel('mu_blood')
ax.set_title('Blood vs Myo intensity at GT offset (each dot = 1 patient)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/contrast_blood_vs_myo_scatter.png', bbox_inches='tight', dpi=120)
plt.show()

C:\Users\Hp\AppData\Local\Temp\ipykernel_26188\1023581129.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_26188\1023581129.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Accuracy Evaluation

In [7]:
from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text

for selector_col, selector_name in [
    ('cnr',            'argmax(cnr)'),
    ('contrast_ratio', 'argmax(contrast_ratio)'),
    ('michelson',      'argmax(michelson)'),
]:
    pred_dict = {}
    for pid in df['patient_id'].unique():
        sub = df[df['patient_id']==pid].sort_values('image_no')
        pred_dict[pid] = int(np.argmax(sub[selector_col].values)) + 1

    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    print(summary_text(selector_name, res['top1'], res['pm1'], res['mae_hz'], res['n_valid']))

print('NB04 complete. Proceed to nb05_banding_detection.ipynb')


  argmax(cnr)
  Top-1 accuracy : 0/25 = 0.0%
  ±1 step        : 1/25 = 4.0%
  MAE            : 114.0 Hz


  argmax(contrast_ratio)
  Top-1 accuracy : 0/25 = 0.0%
  ±1 step        : 2/25 = 8.0%
  MAE            : 110.0 Hz


  argmax(michelson)
  Top-1 accuracy : 0/25 = 0.0%
  ±1 step        : 2/25 = 8.0%
  MAE            : 110.0 Hz

NB04 complete. Proceed to nb05_banding_detection.ipynb
